# Coursera 課程資料清洗與寫入 Supabase

依 **`coursera_cleaning_steps.md`** 逐步清洗 `Coursera_row_rows.csv`，最後寫入 Supabase **course** 表。

**執行方式**：由上而下一個 cell 一個 cell 執行。

**前置條件**：
- 工作目錄為 `supabase_control/course` 或專案根目錄，路徑可依需要調整。
- `.env` 設有 `SUPABASE_URL`、`SUPABASE_SERVICE_ROLE_KEY` 或 `SUPABASE_KEY`。
- Supabase 已建立 **course** 表（見後方 Step 13 的 SQL）。

---
## 環境設定與連線

In [45]:
import os
import re
import json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()
for p in [Path("Erd/.env"), Path(".env"), Path("supabase_control/Erd/.env"), Path("course/.env")]:
    if p.exists():
        load_dotenv(p)
        break

SUPABASE_URL = os.environ.get("SUPABASE_URL")
SUPABASE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY") or os.environ.get("SUPABASE_KEY")
assert SUPABASE_URL and SUPABASE_KEY, "請設定 SUPABASE_URL 與 SUPABASE_KEY（.env）"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# 資料路徑：優先 course 目錄，否則當前目錄
COURSE_DIR = Path.cwd() / "course" if (Path.cwd() / "course").exists() else Path.cwd()
if not (COURSE_DIR / "Coursera_row_rows.csv").exists():
    COURSE_DIR = Path.cwd()
RAW_CSV = COURSE_DIR / "Coursera_row_rows.csv"
print(f"資料目錄: {COURSE_DIR}")
print(f"來源 CSV: {RAW_CSV}")
print(f"Supabase: {SUPABASE_URL[:50]}...")

資料目錄: c:\Users\Elvis\git\final\supabase_control\course
來源 CSV: c:\Users\Elvis\git\final\supabase_control\course\Coursera_row_rows.csv
Supabase: https://nyslsqlgsavvfwiducdu.supabase.co...


---
## Step 1：讀取與基本檢查

In [46]:
df = pd.read_csv(RAW_CSV)
print("shape:", df.shape)
print("columns:", df.columns.tolist())
print("\n缺失:")
print(df.isnull().sum())
df.head(3)

shape: (839, 15)
columns: ['ID', '主要技能名稱', '課程名稱', '評分', '評論數', 'Metadata', '課程網址', '課程', '技能', '課程資訊', '師資', '開課時間', '建議學習時間', '學習時長', '語言']

缺失:
ID            0
主要技能名稱        0
課程名稱          0
評分            0
評論數           0
Metadata      0
課程網址          0
課程            0
技能           52
課程資訊        464
師資          839
開課時間         48
建議學習時間      464
學習時長        464
語言          416
dtype: int64


,ID,主要技能名稱,課程名稱,評分,評論數,Metadata,課程網址,課程,技能,課程資訊,師資,開課時間,建議學習時間,學習時長,語言
0,1,Python,Introduction to Data Analysis Using Python,4.6,11 reviews,Beginner · Course · 1 - 4 Weeks,https://www.coursera.org/learn/introduction-to...,Introduction to Data Analysis Using Python,"Data Structures, Python Programming, Object Or...",NaN,NaN,Starts Feb 17,NaN,NaN,NaN
1,2,Python,"Python for Data Science, AI & Development",4.6,43K reviews,Beginner · Course · 1 - 3 Months,https://www.coursera.org/learn/python-for-appl...,"Python for Data Science, AI & Development","JSON, Pandas (Python Package), File I/O, Objec...",NaN,NaN,Starts Feb 17,NaN,NaN,NaN
2,3,Python,Python for Everybody,4.8,280K reviews,Beginner · Specialization · 3 - 6 Months,https://www.coursera.org/specializations/python,Python for Everybody,"Data Structures, Data Visualization Software, ...",Programming for Everybody (Getting Started wit...,NaN,Starts Feb 17,P2M,P2M,en


---
## Step 2：移除「語言」與「開課時間」

In [47]:
df = df.drop(columns=["語言", "開課時間"], errors="ignore")
print("剩餘欄位:", df.columns.tolist())
df.shape

剩餘欄位: ['ID', '主要技能名稱', '課程名稱', '評分', '評論數', 'Metadata', '課程網址', '課程', '技能', '課程資訊', '師資', '建議學習時間', '學習時長']


(839, 13)

---
## Step 3：刪除重複欄位「課程」

In [48]:
df = df.drop(columns=["課程"], errors="ignore")
df.columns.tolist()

['ID',
 '主要技能名稱',
 '課程名稱',
 '評分',
 '評論數',
 'Metadata',
 '課程網址',
 '技能',
 '課程資訊',
 '師資',
 '建議學習時間',
 '學習時長']

---
## Step 4：評分標準化（rating）

In [49]:
def parse_rating(val):
    if pd.isna(val): return None
    m = re.search(r"(\d+\.?\d*)", str(val))
    if not m: return None
    r = float(m.group(1))
    return min(5.0, r) if r > 5 else r

df["rating"] = df["評分"].apply(parse_rating)
print(df[["評分", "rating"]].head(10))
print("\nrating dtype:", df["rating"].dtype)

    評分  rating
0  4.6     4.6
1  4.6     4.6
2  4.8     4.8
3  4.8     4.8
4  4.8     4.8
5  4.6     4.6
6  4.4     4.4
7  4.7     4.7
8  4.8     4.8
9  4.8     4.8

rating dtype: float64


---
## Step 5：評論數標準化（review_count）

In [50]:
def parse_review_count(val):
    if pd.isna(val): return None
    s = str(val)
    m = re.search(r"([\d,]+(?:\.[\d]+)?)\s*(K|M)?\s*reviews?", s, re.I)
    if not m: return None
    n = float(m.group(1).replace(",", ""))
    k = (m.group(2) or "").upper()
    if k == "K": n *= 1000
    elif k == "M": n *= 1e6
    return int(n)

df["review_count"] = df["評論數"].apply(parse_review_count)
print(df[["評論數", "review_count"]].head(10))
print("\nreview_count 非空:", df["review_count"].notna().sum())

            評論數  review_count
0    11 reviews          11.0
1   43K reviews       43000.0
2  280K reviews      280000.0
3   40K reviews       40000.0
4  233K reviews      233000.0
5  1.8K reviews        1800.0
6   628 reviews         628.0
7   20K reviews       20000.0
8   18K reviews       18000.0
9   23K reviews       23000.0

review_count 非空: 837


---
## Step 6：技能欄標準化（skill_list）

In [51]:
def to_skill_list(x):
    if pd.isna(x) or str(x).strip() == "": return []
    return [s.strip() for s in str(x).split(",") if s.strip()]

df["skill_list"] = df["技能"].apply(to_skill_list)
df["skills"] = df["skill_list"].apply(lambda x: x if isinstance(x, list) else [])
print("範例 skill_list:", df["skill_list"].iloc[0][:5], "...")
df[["主要技能名稱", "skill_list"]].head(3)

範例 skill_list: ['Data Structures', 'Python Programming', 'Object Oriented Programming (OOP)', 'Data Analysis', 'Scripting'] ...


,主要技能名稱,skill_list
0,Python,"[Data Structures, Python Programming, Object O..."
1,Python,"[JSON, Pandas (Python Package), File I/O, Obje..."
2,Python,"[Data Structures, Data Visualization Software,..."


---
## Step 7：Metadata 拆欄（level, course_type）

In [52]:
meta = df["Metadata"].fillna("")
df["level"] = meta.str.extract(r"(Beginner|Intermediate|Advanced)", expand=False)
df["course_type"] = meta.str.extract(
    r"(Course|Specialization|Professional Certificate|Guided Project)", expand=False
)
print("level 分布:", df["level"].value_counts(dropna=False).head())
print("\ncourse_type 分布:", df["course_type"].value_counts(dropna=False).head())
df[["Metadata", "level", "course_type"]].head(5)

level 分布: level
Beginner        494
Intermediate    305
NaN              27
Advanced         13
Name: count, dtype: int64

course_type 分布: course_type
Course                      415
Specialization              281
Professional Certificate     94
Guided Project               37
NaN                          12
Name: count, dtype: int64


,Metadata,level,course_type
0,Beginner · Course · 1 - 4 Weeks,Beginner,Course
1,Beginner · Course · 1 - 3 Months,Beginner,Course
2,Beginner · Specialization · 3 - 6 Months,Beginner,Specialization
3,Beginner · Course · 1 - 3 Months,Beginner,Course
4,Beginner · Course · 1 - 3 Months,Beginner,Course


---
## Step 8：建議學習時間標準化（duration_suggested）

In [53]:
def standardize_duration(row):
    # 先看「建議學習時間」
    sug = row.get("建議學習時間") or row.get("建議學習時間") if "建議學習時間" in row.index else None
    if pd.notna(sug) and str(sug).strip():
        s = str(sug).strip()
        m = re.match(r"P(\d+)M", s, re.I)
        if m: return f"{m.group(1)} months"
        if "month" in s.lower() or "months" in s.lower(): return s
        if "week" in s.lower() or "weeks" in s.lower(): return s
        if "hour" in s.lower(): return s
        return s
    # 從 Metadata 擷取
    meta = str(row.get("Metadata", ""))
    if "Less Than 2 Hours" in meta: return "< 2 hours"
    m = re.search(r"(\d+)\s*-\s*(\d+)\s*(Weeks?|Months?)", meta, re.I)
    if m: return f"{m.group(1)}-{m.group(2)} {m.group(3).lower()}"
    m = re.search(r"(\d+)\s*(Weeks?|Months?)", meta, re.I)
    if m: return f"{m.group(1)} {m.group(2).lower()}"
    return None

df["duration_suggested"] = df.apply(standardize_duration, axis=1)
print("duration_suggested 範例:")
print(df[["建議學習時間", "Metadata", "duration_suggested"]].drop_duplicates("duration_suggested").head(15))

duration_suggested 範例:
    建議學習時間                                           Metadata  \
0      NaN                    Beginner · Course · 1 - 4 Weeks   
1      NaN                   Beginner · Course · 1 - 3 Months   
2      P2M           Beginner · Specialization · 3 - 6 Months   
6      P4M  Beginner · Professional Certificate · 3 - 6 Mo...   
9      P3M           Beginner · Specialization · 3 - 6 Months   
11     P6M  Advanced · Professional Certificate · 3 - 6 Mo...   
17     P1M       Intermediate · Specialization · 1 - 3 Months   
23     P7M  Beginner · Professional Certificate · 3 - 6 Mo...   
43     NaN      Beginner · Guided Project · Less Than 2 Hours   
48     P5M           Beginner · Specialization · 3 - 6 Months   
75     NaN                   Advanced · Course · 3 - 6 Months   
98     P8M  Beginner · Professional Certificate · 3 - 6 Mo...   
559    P9M  Beginner · Professional Certificate · 3 - 6 Mo...   
574    NaN                               Degree · 1 - 4 Years   

 

---
## Step 10：去重（以課程網址為唯一鍵）

In [54]:
before = len(df)
df = df.drop_duplicates(subset=["課程網址"], keep="first")
print(f"去重前 {before} 筆，去重後 {len(df)} 筆")

去重前 839 筆，去重後 665 筆


---
## Step 11：缺失值與型別、整理輸出欄位

In [55]:
# 必填：課程名稱、課程網址 不得為空
df = df.dropna(subset=["課程名稱", "課程網址"])

# 輸出用 DataFrame：對齊 DB 欄位（snake_case）
out = pd.DataFrame()
out["course_name"] = df["課程名稱"].astype(str)
out["url"] = df["課程網址"].astype(str)
out["primary_skill_name"] = df["主要技能名稱"].fillna("").astype(str)
out["rating"] = pd.to_numeric(df["rating"], errors="coerce")
out["review_count"] = pd.to_numeric(df["review_count"], errors="coerce").astype("Int64")
out["level"] = df["level"]
out["course_type"] = df["course_type"]
out["course_information"] = df["課程資訊"].fillna("").astype(str)
out["duration_suggested"] = df["duration_suggested"]
out["skills"] = df["skills"].apply(lambda x: x if isinstance(x, list) else [])
out["source_platform"] = "Coursera"

print("輸出筆數:", len(out))
print(out.dtypes)
out.head()

輸出筆數: 665
course_name               str
url                       str
primary_skill_name        str
rating                float64
review_count            Int64
level                     str
course_type               str
course_information        str
duration_suggested        str
skills                 object
source_platform           str
dtype: object


,course_name,url,primary_skill_name,rating,review_count,level,course_type,course_information,duration_suggested,skills,source_platform
0,Introduction to Data Analysis Using Python,https://www.coursera.org/learn/introduction-to...,Python,4.6,11,Beginner,Course,,1-4 weeks,"[Data Structures, Python Programming, Object O...",Coursera
1,"Python for Data Science, AI & Development",https://www.coursera.org/learn/python-for-appl...,Python,4.6,43000,Beginner,Course,,1-3 months,"[JSON, Pandas (Python Package), File I/O, Obje...",Coursera
2,Python for Everybody,https://www.coursera.org/specializations/python,Python,4.8,280000,Beginner,Specialization,Programming for Everybody (Getting Started wit...,2 months,"[Data Structures, Data Visualization Software,...",Coursera
3,Crash Course on Python,https://www.coursera.org/learn/python-crash-co...,Python,4.8,40000,Beginner,Course,,1-3 months,"[Problem Management, Development Environment, ...",Coursera
4,Programming for Everybody (Getting Started wit...,https://www.coursera.org/learn/python,Python,4.8,233000,Beginner,Course,,1-3 months,"[Software Installation, Computational Thinking...",Coursera


---
## Step 12：匯出清洗後 CSV（可選）

In [ ]:
# # skills 存成 JSON 字串以便 CSV 單欄
# out_csv = out.copy()
# out_csv["skills"] = out_csv["skills"].apply(json.dumps)
# out_csv.to_csv(COURSE_DIR / "courses_cleaned.csv", index=False, encoding="utf-8-sig")
# print(f"已寫入 {COURSE_DIR / 'courses_cleaned.csv'}")

已寫入 c:\Users\Elvis\git\final\supabase_control\course\courses_cleaned.csv


---
## Step 13：建立新表與關聯鍵（Supabase SQL Editor）

在 Supabase **SQL Editor** 執行以下 SQL，建立 **course** 表。若已有表可跳過。

In [57]:
CREATE_TABLE_SQL = """
-- 課程主表（對齊 coursera_cleaning_steps.md Step 13）
CREATE TABLE IF NOT EXISTS course (
  course_id BIGSERIAL PRIMARY KEY,
  course_name VARCHAR(500) NOT NULL,
  url VARCHAR(500) NOT NULL UNIQUE,
  primary_skill_name VARCHAR(100),
  primary_skill_id INT REFERENCES skill_master(skill_id),
  rating NUMERIC(3,2),
  review_count INT,
  level VARCHAR(50),
  course_type VARCHAR(100),
  course_information TEXT,
  duration_suggested VARCHAR(100),
  skills JSONB,
  source_platform VARCHAR(50) DEFAULT 'Coursera',
  created_at TIMESTAMPTZ DEFAULT now()
);

-- 若 skill_master 表名為 snake_case，且尚未有 primary_skill_id 可先省略 FK：
-- 建立後可再加: ALTER TABLE course ADD CONSTRAINT fk_course_primary_skill
--   FOREIGN KEY (primary_skill_id) REFERENCES skill_master(skill_id);
"""
print(CREATE_TABLE_SQL)
print("請複製以上 SQL 到 Supabase SQL Editor 執行。若表已存在可跳過。")


-- 課程主表（對齊 coursera_cleaning_steps.md Step 13）
CREATE TABLE IF NOT EXISTS course (
  course_id BIGSERIAL PRIMARY KEY,
  course_name VARCHAR(500) NOT NULL,
  url VARCHAR(500) NOT NULL UNIQUE,
  primary_skill_name VARCHAR(100),
  primary_skill_id INT REFERENCES skill_master(skill_id),
  rating NUMERIC(3,2),
  review_count INT,
  level VARCHAR(50),
  course_type VARCHAR(100),
  course_information TEXT,
  duration_suggested VARCHAR(100),
  skills JSONB,
  source_platform VARCHAR(50) DEFAULT 'Coursera',
  created_at TIMESTAMPTZ DEFAULT now()
);

-- 若 skill_master 表名為 snake_case，且尚未有 primary_skill_id 可先省略 FK：
-- 建立後可再加: ALTER TABLE course ADD CONSTRAINT fk_course_primary_skill
--   FOREIGN KEY (primary_skill_id) REFERENCES skill_master(skill_id);

請複製以上 SQL 到 Supabase SQL Editor 執行。若表已存在可跳過。


---
## Step 14 / 15：寫入 Supabase（upsert，以 url 為唯一鍵）

第一次寫入即帶入 **primary_skill_id**（依 skill_master 對照）。以 **url** 為唯一鍵 upsert：同 URL 會更新、不重複插入，之後有新資料重跑即可覆寫。

In [58]:
# 1. 從 skill_master 建立「名稱（含同義詞）→ skill_id」對照，第一次寫入就帶入 primary_skill_id
try:
    sm_resp = supabase.table("skill_master").select("skill_id, skill_name, synonyms").limit(5000).execute()
    sm_df = pd.DataFrame(sm_resp.data or [])
    name_to_skill_id = {}
    for _, row in sm_df.iterrows():
        sid = row["skill_id"]
        name = str(row["skill_name"]).strip()
        syn = row.get("synonyms")
        syn_list = []
        if syn is not None:
            if isinstance(syn, str):
                try: syn_list = json.loads(syn)
                except Exception: pass
            else: syn_list = list(syn) if syn else []
        name_to_skill_id[name.lower()] = sid
        for s in syn_list:
            if s and str(s).strip(): name_to_skill_id[str(s).strip().lower()] = sid
    print(f"✅ skill_master 對照表 {len(name_to_skill_id)} 個名稱/同義詞")
except Exception as e:
    print(f"⚠️ 無法讀取 skill_master（{e}），primary_skill_id 將為空")
    name_to_skill_id = {}

# 2. 組 payload，並帶入 primary_skill_id
rows = out.copy()
rows["skills"] = rows["skills"].apply(lambda x: x if isinstance(x, list) else [])
rows["primary_skill_id"] = rows["primary_skill_name"].fillna("").astype(str).str.strip().str.lower().map(name_to_skill_id)
payload = rows.to_dict(orient="records")
# NaN 轉 None，否則 JSON 會報錯；list/dict/array 不適用 pd.isna，跳過或 try 捕獲
for r in payload:
    for k, v in r.items():
        if isinstance(v, (list, dict)):
            continue
        try:
            if pd.isna(v):
                r[k] = None
        except (TypeError, ValueError):
            pass

# 3. 以 url 為唯一鍵 upsert：同 URL 會更新不重複插入，新資料可重跑此 cell 覆寫
BATCH = 100
for i in range(0, len(payload), BATCH):
    batch = payload[i : i + BATCH]
    try:
        supabase.table("course").upsert(batch, on_conflict="url").execute()
        print(f"Upsert 第 {i+1}～{min(i+BATCH, len(payload))} 筆")
    except Exception as e:
        print(f"批次 {i} 錯誤: {e}")
        raise

print("\n寫入完成。（以 url 為唯一鍵，重跑會更新同 URL 的列，不會重複插入）")

✅ skill_master 對照表 135 個名稱/同義詞
Upsert 第 1～100 筆
Upsert 第 101～200 筆
Upsert 第 201～300 筆
Upsert 第 301～400 筆
Upsert 第 401～500 筆
Upsert 第 501～600 筆
Upsert 第 601～665 筆

寫入完成。（以 url 為唯一鍵，重跑會更新同 URL 的列，不會重複插入）


---
## 驗證：查詢 course 表

In [63]:
resp = supabase.table("course").select("course_id, course_name, url, rating, review_count", count="exact").limit(5).execute()
print("總筆數:", resp.count)
pd.DataFrame(resp.data)

總筆數: 665


,course_id,course_name,url,rating,review_count
0,2,"Python for Data Science, AI & Development",https://www.coursera.org/learn/python-for-appl...,4.6,43000
1,4,Crash Course on Python,https://www.coursera.org/learn/python-crash-co...,4.8,40000
2,134,Single Page Web Applications with AngularJS,https://www.coursera.org/learn/single-page-web...,4.8,1900
3,17,"JavaScript Programming with React, Node & MongoDB",https://www.coursera.org/specializations/javas...,4.4,1400
4,18,Advanced JavaScript,https://www.coursera.org/specializations/advan...,4.6,17
